# Can we predict late deliveries?
Target: `late_delivery` (1 = delivered after promised date). Goal: decide whether features known **at dispatch** carry enough signal to predict lateness.

Plan: audit data & leakage, EDA, time-based split, baseline vs. models, evaluation (ROC AUC, PR AUC, top-k lift), conclusions.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve, brier_score_loss
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv',parse_dates=['dispatched_on'])
print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data audit

In [ ]:
print(df.shape, 'duplicate ids:', df.consignment.duplicated().sum(), 'missing:', df.isna().sum().sum())
print(df.dispatched_on.min(), '->', df.dispatched_on.max())
display(df.describe().T)
print(df.late_delivery.value_counts(normalize=True).rename('share'))

## 2. Leakage check: `claim_paid_eur`
A claim is settled *after* delivery, so it is not available when a prediction would be made. Let's see how it relates to the target.

In [ ]:
df['has_claim']=df.claim_paid_eur>0
print(df.groupby('has_claim').late_delivery.agg(['mean','size']))
print('AUC of claim_paid_eur alone:', round(roc_auc_score(df.late_delivery, df.claim_paid_eur),3))

`claim_paid_eur` is essentially a downstream consequence of the outcome: no claims are paid on on-time consignments in the table above, and ~90% of claim-bearing consignments are late. It would give an impressive but **useless** model because the value is unknown at dispatch. **It is excluded from all models below.** Usable features: `carrier`, `distance_km`, `weight_kg`, `parcels`, and the dispatch timestamp.

## 3. EDA

In [ ]:
e=df.assign(hour=df.dispatched_on.dt.hour, dow=df.dispatched_on.dt.dayofweek, month=df.dispatched_on.dt.to_period('M').dt.to_timestamp())
fig,ax=plt.subplots(2,3,figsize=(15,8))
e.groupby('carrier').late_delivery.mean().plot.bar(ax=ax[0,0],title='Late rate by carrier')
e.groupby(pd.qcut(e.distance_km,10,duplicates='drop'),observed=True).late_delivery.mean().plot(ax=ax[0,1],marker='o',title='Late rate by distance decile'); ax[0,1].tick_params(axis='x',rotation=45)
e.groupby(pd.qcut(e.weight_kg,10,duplicates='drop'),observed=True).late_delivery.mean().plot(ax=ax[0,2],marker='o',title='Late rate by weight decile'); ax[0,2].tick_params(axis='x',rotation=45)
e.groupby('parcels').late_delivery.mean().plot(ax=ax[1,0],marker='o',title='Late rate by parcels')
e.groupby('hour').late_delivery.mean().plot(ax=ax[1,1],marker='o',title='Late rate by dispatch hour')
e.groupby('month').late_delivery.mean().plot(ax=ax[1,2],marker='o',title='Late rate by month')
plt.tight_layout(); plt.show()
print(e.groupby('dow').late_delivery.mean().round(3).to_string())
print(e.groupby('month').size().to_string())

## 4. Time drift: the late rate is not stable
The monthly late rate swings between ~1% and ~49%, and drops to 1-5% from Oct 2025 onward. A randomly or naively chronologically split test set would mostly measure this drift, not the features' skill.

In [ ]:
m=df.dispatched_on.dt.to_period('M')
mr=df.groupby(m).late_delivery.mean()
mr.plot(marker='o',figsize=(10,3.5),title='Monthly late rate'); plt.axvline(pd.Period('2025-10'),color='r',ls='--'); plt.show()
print(df.groupby([m,'carrier']).late_delivery.mean().unstack().round(2).tail(8))
print('claim share Oct25-Feb26:', round((df[df.dispatched_on>='2025-10-01'].claim_paid_eur>0).mean(),3))

Feature distributions (distance, weight, parcels) are flat over time, so the rate shift is not explained by the inputs. Possible causes: seasonal/operational effects we have no columns for, or **label immaturity** (recent consignments not yet fully resolved, so `late_delivery=0` by default). We can't tell from this data alone, so we should confirm with the data owner. Meanwhile:
- **Main evaluation**: consignments dispatched before 2025-10-01, split chronologically 75/25.
- **Stress test**: the model trained on that window, scored on Oct 2025 - Feb 2026.

## 5. Features and time-based split

In [ ]:
df=df.sort_values('dispatched_on').reset_index(drop=True)
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek
df['weekend']=(df.dow>=5).astype(int)
df['kg_per_parcel']=df.weight_kg/df.parcels
num=['distance_km','weight_kg','parcels','kg_per_parcel','hour','dow','weekend']; cat=['carrier']
mature=df[df.dispatched_on<'2025-10-01']; recent=df[df.dispatched_on>='2025-10-01']
cut=int(len(mature)*0.75)
tr,te=mature.iloc[:cut],mature.iloc[cut:]
Xtr,ytr,Xte,yte=tr[num+cat],tr.late_delivery,te[num+cat],te.late_delivery
Xrc,yrc=recent[num+cat],recent.late_delivery
print('train',tr.dispatched_on.min(),'->',tr.dispatched_on.max(),'| late rate',round(ytr.mean(),3))
print('test ',te.dispatched_on.min(),'->',te.dispatched_on.max(),'| late rate',round(yte.mean(),3))
print('recent',recent.dispatched_on.min(),'->',recent.dispatched_on.max(),'| late rate',round(yrc.mean(),3))

## 6. Models

In [ ]:
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':Pipeline([('p',pre),('m',LogisticRegression(max_iter=1000))]),
 'grad. boosting':Pipeline([('p',pre),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=0))]),
}
rows=[];proba={}
for n,m in models.items():
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; proba[n]=p; pr_=m.predict_proba(Xrc)[:,1]
    rows.append(dict(model=n,roc_auc=roc_auc_score(yte,p),pr_auc=average_precision_score(yte,p),brier=brier_score_loss(yte,p),
                     recent_roc_auc=roc_auc_score(yrc,pr_),recent_mean_pred=pr_.mean()))
display(pd.DataFrame(rows).set_index('model').round(3))
print('Test late rate (= PR AUC of a no-skill model):',round(yte.mean(),3),'| recent late rate:',round(yrc.mean(),3))

In [ ]:
# uncertainty: bootstrap CI on test AUC
rng=np.random.default_rng(0); yt=yte.values
for n in ['logistic','grad. boosting']:
    p=proba[n]; a=[]
    for _ in range(500):
        i=rng.integers(0,len(yt),len(yt)); a.append(roc_auc_score(yt[i],p[i]))
    print(n,'AUC 95% CI:',np.percentile(a,[2.5,97.5]).round(3))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for n in ['logistic','grad. boosting']:
    f,t,_=roc_curve(yte,proba[n]); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,proba[n]); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k--',label='chance'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR'); ax[0].legend()
ax[1].axhline(yte.mean(),color='k',ls='--',label='no skill'); ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision'); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. What drives the predictions?

In [ ]:
m=models['grad. boosting']
pi=permutation_importance(m,Xte,yte,scoring='roc_auc',n_repeats=10,random_state=0)
imp=pd.Series(pi.importances_mean,index=Xte.columns).sort_values()
imp.plot.barh(xerr=pd.Series(pi.importances_std,index=Xte.columns)[imp.index],title='Permutation importance (drop in test AUC)'); plt.show()
lr=models['logistic']; names=lr['p'].get_feature_names_out()
print(pd.Series(lr['m'].coef_[0],index=names).sort_values().round(3).to_string())

## 8. Practical value: flagging the riskiest consignments

In [ ]:
p=proba['grad. boosting']; base=yte.mean(); out=[]
for q in [0.1,0.2,0.3,0.5]:
    k=int(len(p)*q); idx=np.argsort(-p)[:k]
    out.append(dict(flag_top=f'{int(q*100)}%',late_rate_in_flagged=yte.values[idx].mean(),lift=yte.values[idx].mean()/base,share_of_all_late_caught=yte.values[idx].sum()/yte.sum()))
display(pd.DataFrame(out).round(3))

## 9. Conclusions
(Numbers from the run on this data; re-running reproduces them, all seeds are fixed.)

- **Yes, but only moderately.** On the chronological test window (May-Sep 2025, 29% late) logistic regression / gradient boosting reach ROC AUC ~0.70-0.71 (95% bootstrap CI ~0.67-0.74) and PR AUC ~0.47-0.49 vs. 0.29 for no skill. Flagging the riskiest 20% of consignments catches ~35% of late ones at ~51% precision (1.8x lift).
- **Signal comes from distance, weight and carrier** (longer, heavier, `coastal` = later). Dispatch hour, weekday, parcel count add essentially nothing. A simple logistic regression does as well as boosting, so there is no evidence of complex interactions.
- **`claim_paid_eur` must not be used.** It looks nearly perfect (AUC 0.99) only because claims are paid *because* a delivery was late; it is unknown at dispatch.
- **The biggest caveat is time.** The late rate varies from ~1% to ~49% by month and falls to ~2.5% after Oct 2025, none of which the available features explain. The model keeps ranking reasonably in that period (AUC ~0.68-0.71) but its probabilities are badly miscalibrated (mean prediction ~24% vs. 2.5% actual). Use it for ranking/prioritisation, not for absolute probabilities, unless it is recalibrated on recent data.
- **To check with the data owner:** whether the Oct 2025+ drop is real (process change) or label immaturity. Adding operational drivers (weather, depot load, promised lead time, destination region) is the most likely route to better accuracy.